In [2]:

"""
Rainbow Çok Katmanlı İmza Şeması: Anahtar Üretimi, İmzalama ve Doğrulama
==========================================================================

Bu modül, Oil and Vinegar imza şemasının çok katmanlı genellemesi olan
RAINBOW imza şemasının öğretici bir SageMath simülasyonunu içermektedir.

Rainbow'un Oil-Vinegar'dan Farkı Nedir?
-----------------------------------------
Klasik (tek katmanlı) Oil-Vinegar şemasında değişkenler yalnızca İKİ
gruba ayrılır: Vinegar (v adet) ve Oil (o adet); merkezi dönüşümün tek
bir kısıtlaması vardır (Oil x Oil çarpraz terimlerin yasak olması).
RAINBOW ise bu fikri KADEMELİ (katmanlı) hale getirir: bir önceki
katmanın Oil değişkenleri, bir SONRAKİ katmanın Vinegar değişkenleri
olarak yeniden kullanılır [Ding-Schmidt, 2005]. Bu yapı şu avantajı
sağlar: her katman kendi içinde bir Oil-Vinegar alt problemi olduğundan,
imzalama sırasında sistem KATMAN KATMAN (bir öncekinin çözümü bir
sonrakine "bilinen" olarak aktarılarak) çözülebilir; bu da tek katmanlı
OV'ye kıyasla daha büyük Oil/Vinegar oranları ve daha verimli parametre
seçimleri sunar.

Katman Yapısının Matematiksel Temsili
----------------------------------------
`v_layers = [v_0, v_1, ..., v_u]` listesi, kümülatif değişken
sınırlarını tanımlar:
    - x_0, ..., x_{v_0 - 1}         : Başlangıç Vinegar değişkenleri
                                       (hiçbir katmanda Oil olarak
                                       çözülmez, yalnızca rastgele
                                       seçilir).
    - Katman i (i = 1, ..., u)       : Vinegar kümesi = x_0, ..., x_{v_{i-1}-1}
                                        (önceki tüm değişkenler);
                                        Oil kümesi = x_{v_{i-1}}, ..., x_{v_i - 1}.
Her katmanın merkezi denklemlerinde, o katmanın Oil değişkenleri
KENDİ ARALARINDA çarpılamaz (Oil x Oil bloğu sıfır); ancak Vinegar
değişkenleriyle (kendi katmanının Vinegar'ı = önceki tüm değişkenler)
serbestçe çarpılabilirler. Bu, tek katmanlı OV'deki "Oil x Oil yasak"
kuralının doğrudan kademeli genellemesidir.

Rainbow'un Trapdoor Mekanizması
----------------------------------
Genel akış, klasik çok değişkenli kuadratik (MQ) imza şemalarıyla
aynıdır: gizli bir merkezi dönüşüm F (katman yapısı sayesinde kolay
tersine çevrilebilir) iki gizli AFİN (doğrusal + öteleme) dönüşüm S ve
T arasına sıkıştırılarak açık anahtar
    P(x) = S( F( T(x) ) )
elde edilir. İmzalama, hash edilmiş mesajı S^{-1} ile "geri çekip",
F'yi katman katman (her katmanda Vinegar değişkenlerini rastgele seçip
Oil değişkenleri için ORTAYA ÇIKAN DOĞRUSAL sistemi çözerek) tersine
çevirmeyi ve son olarak T^{-1} uygulamayı gerektirir. Doğrulama ise
yalnızca açık anahtar polinomlarının imza noktasında değerlendirilip
mesajla karşılaştırılmasından ibarettir (tek yönlü kapı fonksiyonunun
"ileri" -- yani kolay -- yönü).

Bu Simülasyonun Sınırları ve Amaçları
-----------------------------------------
- Bu kod GÜVENLİ parametreler üretmek için değil, Rainbow'un katmanlı
  yapısının cebirsel işleyişini (merkezi dönüşümün blok kısıtlaması,
  katman katman doğrusal çözüm stratejisi, afin maskeleme) somut
  biçimde göstermek amacıyla hazırlanmıştır.
- Küçük bir sonlu cisim (örn. GF(7)) ve az sayıda katman/değişken
  üzerinde çalıştırılarak ara adımların (matrisler, polinomlar,
  denklem sistemleri) okunabilir biçimde ekrana yazdırılması
  hedeflenmiştir.
- İmzalama fonksiyonu (sign), her denemede rastgele Vinegar değerleri
  seçtiğinden BAŞARISIZ olabilir (ortaya çıkan doğrusal sistem
  tekil/çözümsüz olabilir); bu nedenle en fazla 100 deneme yapılan bir
  RETRY mekanizması içerir -- bu, Oil-Vinegar ailesi şemalarının
  DOĞASINDA olan, kriptografik güvenliği etkilemeyen normal bir
  durumdur.

Algoritmanın Genel Akışı
--------------------------
1. Anahtar Üretimi (generate_keys):
   Gizli afin dönüşümler (T, S) ve katman katman kısıtlanmış merkezi
   dönüşüm (Central_Polys) rastgele üretilir; bunlardan açık anahtar
   (Public_Key) P(x) = S(F(T(x))) bileşimiyle türetilir.

2. İmzalama (sign):
   Hedef mesaj S^{-1} ile geri çekilir; rastgele başlangıç Vinegar
   değerleriyle başlanıp her katmanda ortaya çıkan doğrusal sistem
   çözülerek merkezi çözüm z inşa edilir; son olarak T^{-1} uygulanarak
   gerçek imza elde edilir.

3. Doğrulama (verify):
   İmza, açık anahtar polinomlarında doğrudan değerlendirilir ve sonuç
   orijinal mesajla karşılaştırılır.

Bu implementasyon; yüksek lisans tezi kapsamında Rainbow imza şemasının
çok katmanlı yapısını ve trapdoor mekanizmasını somutlaştırmak amacıyla
eğitim/araştırma amaçlı geliştirilmiştir.

-------------
Bu kod SageMath ortamında çalıştırılmak üzere tasarlanmıştır ve sonlu
cisimler (GF), çok değişkenli polinom halkaları (PolynomialRing), matris
cebiri (tersinirlik testi, doğrusal sistem çözümü) ve vektör işlemleri
gibi SageMath'in yerleşik simgesel/sayısal cebir araçlarına dayanır.

Referans: Tez, Bölüm 6


"""

from sage.all import *

class RainbowSignatureScheme:
    """
    Rainbow çok katmanlı çok değişkenli kuadratik (MQ) imza şemasının
    öğretici SageMath simülasyonu.

    Bu sınıf; katman katman kısıtlanmış bir gizli merkezi dönüşümü, bu
    dönüşümü maskeleyen iki gizli afin dönüşümü ve bunlardan türetilen
    açık anahtarı üretir; ardından katman katman doğrusal sistem çözümü
    ile imzalama ve açık anahtar değerlendirmesiyle doğrulama
    işlemlerini gerçekleştirir.

    Matematiksel Kurulum
    ---------------------
    - F_q = GF(q) : Temel sonlu cisim.
    - R = F_q[x_0, ..., x_{n-1}] : n değişkenli, F_q katsayılı çok
      değişkenli polinom halkası (n = toplam değişken sayısı).
    - v_layers = [v_0, ..., v_u] : Kümülatif katman sınırları; v_0
      başlangıç Vinegar sayısını, v_u = n toplam değişken sayısını
      belirtir. Katman i'nin Oil kümesi [v_{i-1}, v_i) aralığındaki
      değişkenlerden, Vinegar kümesi ise [0, v_{i-1}) aralığındaki
      (önceki tüm) değişkenlerden oluşur.
    - m = n - v_0 : Toplam denklem (ve dolayısıyla açık anahtar
      bileşeni) sayısı; her katmanın Oil değişken sayısı kadar denklem
      üretilir ve bunların toplamı m'ye eşittir.

    Parametreler
    ------------
    q : int
        Temel sonlu cismin eleman sayısı (asal veya asal kuvveti).
    v_layers : list of int
        Kümülatif katman sınırlarını belirten, artan tam sayılardan
        oluşan liste (örn. [2, 4, 6] -> Katman1: x0,x1 Vinegar, x2,x3
        Oil; Katman2: x0..x3 Vinegar, x4,x5 Oil).

    Öznitelikler (Attributes)
    --------------------------
    n : int
        Toplam değişken sayısı (v_layers listesinin son elemanı).
    v1 : int
        Başlangıç (0. katman) Vinegar değişkenlerinin sayısı
        (v_layers listesinin ilk elemanı).
    m : int
        Toplam denklem sayısı (n - v1).
    F : FiniteField
        Temel sonlu cisim F_q = GF(q).
    R : MPolynomialRing
        n değişkenli, F_q katsayılı çok değişkenli polinom halkası.
    vars : vector
        R halkasının üreteçlerinden oluşan vektör.
    layers : list of dict
        Her katmana ait 'id' (katman numarası), 'v_indices' (Vinegar
        değişken indisleri), 'o_indices' (Oil değişken indisleri) ve
        'num_eqs' (bu katmanda üretilecek denklem sayısı) bilgilerini
        içeren sözlüklerden oluşan liste.
    Central_Polys : list of MPolynomial
        Gizli merkezi dönüşüm F'yi oluşturan, katman katman kısıtlanmış
        m adet polinom.
    Public_Key : list of MPolynomial
        Açık anahtarı oluşturan m adet polinom (P(x) = S(F(T(x)))).
    T_mat, T_vec : Matrix, vector
        Girdi tarafındaki gizli afin dönüşümün (T) sırasıyla doğrusal
        ve öteleme bileşenleri (n x n boyutunda).
    S_mat, S_vec : Matrix, vector
        Çıktı tarafındaki gizli afin dönüşümün (S) sırasıyla doğrusal
        ve öteleme bileşenleri (m x m boyutunda).
    """
    def __init__(self, q, v_layers):
        """
        Rainbow şemasının parametrelerini, temel sonlu cismi, çok
        değişkenli polinom halkasını ve katman yapısını başlatır.

        Parametreler
        ------------
        q : int
            Temel sonlu cismin eleman sayısı.
        v_layers : list of int
            Kümülatif katman sınırları [v_0, v_1, ..., v_u]. v_0,
            başlangıç Vinegar değişkenlerinin sayısını; v_u ise toplam
            değişken sayısını (n) belirtir. Örneğin [2, 4, 5] için
            Katman 1: Vinegar={x0,x1}, Oil={x2,x3}; Katman 2:
            Vinegar={x0,x1,x2,x3}, Oil={x4}.

        Yan Etkiler
        -----------
        self.F, self.R, self.vars, self.layers, self.n, self.v1,
        self.m öznitelikleri hesaplanır; self.Central_Polys ve
        self.Public_Key boş listeler, self.T_mat/self.T_vec/
        self.S_mat/self.S_vec None olarak başlatılır (bunlar ancak
        generate_keys() çağrıldığında doldurulur). Parametre özeti
        ekrana yazdırılır.

        Dönüş Değeri
        ------------
        None
        """
        self.q = q
        self.v_layers = v_layers
        self.n = v_layers[-1]        # Toplam Değişken
        self.v1 = v_layers[0]        # İlk Vinegar Seti
        self.m = self.n - self.v1    # Toplam Denklem

        # 1. Temel Cisim ve Halka
        # F_q = GF(q) temel sonlu cismi ve bu cisim üzerinde n
        # değişkenli çok değişkenli polinom halkası R = F_q[x_0,...,x_{n-1}];
        # Rainbow'un tüm cebirsel yapısı (merkezi dönüşüm, açık anahtar,
        # doğrulama denklemleri) bu halka üzerinde tanımlanır.
        self.F = GF(q, 'a')
        self.R = PolynomialRing(self.F, self.n, 'x')
        self.vars = vector(self.R, self.R.gens())

        print("\n" + "="*70)
        print(f"RAINBOW ŞEMASI: MATRİS VE POLİNOM ANALİZİ")
        print(f"Parametreler: GF({q})")
        print(f"Katman Yapısı: {v_layers} (n={self.n}, m={self.m})")
        print("="*70)

        # Katmanları Hazırla
        # Her katman, kendi Vinegar (önceki tüm değişkenler) ve Oil
        # (bu katmana özgü yeni değişkenler) kümelerini tanımlayan bir
        # sözlük olarak saklanır; bu yapı hem generate_keys() içinde
        # merkezi dönüşümün blok kısıtlamasını uygularken, hem de
        # sign() içinde katman katman doğrusal çözüm sırasında
        # kullanılır.
        self.layers = []
        for i in range(len(v_layers) - 1):
            v_end = v_layers[i]     # Bu katman için "Bilinenler" (Vinegar) sınırı
            o_end = v_layers[i+1]   # Bu katman için "Bilinmeyenler" (Oil) sınırı
            # Oil değişkenleri: [v_end ... o_end-1]
            self.layers.append({
                'id': i+1,
                'v_indices': range(v_end),      # 0'dan v_end-1'e kadar
                'o_indices': range(v_end, o_end), # v_end'den o_end-1'e kadar
                'num_eqs': o_end - v_end
            })

        self.Central_Polys = []
        self.Public_Key = []
        self.T_mat = None; self.T_vec = None
        self.S_mat = None; self.S_vec = None

    def get_poly_matrix(self, poly):
        """
        Bir polinomun kuadratik kısmı için üst üçgensel katsayı matrisini çıkarır.
        Lineer ve sabit terimler bu matrise dahil değildir.

        Teorik Gerekçe
        ---------------
        Bir çok değişkenli kuadratik polinomun matris temsili tek
        değildir (simetrik veya üst üçgensel gösterim tercih
        edilebilir); bu metot ÜST ÜÇGENSEL gösterimi seçer: yalnızca
        i <= j için x_i*x_j teriminin katsayısı M[i,j] hücresine
        yazılır, alt üçgen sıfır bırakılır. Bu temsil, açık anahtar
        polinomlarının (Public_Key) blok yapısını (örn. Oil x Oil
        bloğunun T ve S dönüşümleri altında nasıl "karıştığını")
        görsel olarak incelemek amacıyla generate_keys() içinde
        kullanılır.

        Parametreler
        ------------
        poly : MPolynomial
            Katsayı matrisi çıkarılacak, self.R halkasına ait homojen
            kuadratik (veya kuadratik + lineer + sabit) polinom.

        Dönüş Değeri
        ------------
        Matrix
            n x n boyutunda, F_q üzerinde tanımlı üst üçgensel matris;
            M[i,j] (i<=j), poly içindeki x_i*x_j teriminin katsayısını
            verir.
        """
        M = matrix(self.F, self.n, self.n)
        # Üst üçgensel temsil: i <= j için x_i x_j katsayısı M[i,j] içine yazılır.
        for i in range(self.n):
            for j in range(self.n):
                # x_i * x_j katsayısı
                # Simetrik değil üst üçgen olarak dolduralım
                if i <= j:
                    val = poly.monomial_coefficient(self.vars[i] * self.vars[j])
                    M[i, j] = val
        return M


    def _check_keys_generated(self):
        """
        Açık ve gizli anahtarların üretilip üretilmediğini kontrol eder.

        Bu iç yardımcı metot, sign() ve verify() metotlarının
        generate_keys() çağrılmadan kullanılmasını engelleyen bir
        koruma (guard) katmanıdır; T_mat, S_mat, Central_Polys veya
        Public_Key bileşenlerinden herhangi biri eksikse anlamlı bir
        hata mesajıyla erken durdurma sağlar.

        Ön Koşullar
        ------------
        Yoktur (bu metot herhangi bir zamanda çağrılabilir; amacı tam
        olarak diğer metotların ön koşulunu denetlemektir).

        Dönüş Değeri
        ------------
        None
            Anahtarlar eksiksizse sessizce döner.

        Fırlatılan İstisnalar
        -----------------------
        RuntimeError
            self.T_mat veya self.S_mat None ise, ya da
            self.Central_Polys veya self.Public_Key boşsa fırlatılır.
        """
        if self.T_mat is None or self.S_mat is None or len(self.Central_Polys) == 0 or len(self.Public_Key) == 0:
            raise RuntimeError("Önce generate_keys() çağrılmalıdır.")


    def generate_keys(self):
        """
        Rainbow şemasının gizli anahtarını (afin dönüşümler T, S ve
        katman katman kısıtlanmış merkezi dönüşüm F) rastgele üretir
        ve bunlardan açık anahtarı (Public_Key) türetir.

        Algoritmanın Adımları
        -----------------------
        1. Gizli Afin Dönüşümler (T, S):
           Girdi tarafında n x n boyutunda TERSİNİR rastgele bir T_mat
           ve rastgele bir öteleme vektörü T_vec seçilir; çıktı
           tarafında ise m x m boyutunda TERSİNİR rastgele bir S_mat
           ve rastgele bir öteleme vektörü S_vec seçilir. Bu iki afin
           dönüşüm, merkezi dönüşümün katmanlı blok yapısını
           saldırgandan gizleyen maskeleme katmanlarıdır (T girdi
           değişkenlerini, S ise çıktı denklemlerini karıştırır).

        2. Merkezi Dönüşüm (F), Katman Katman:
           Her katman için, o katmanın Oil değişken sayısı kadar
           polinom üretilir. Her polinomun kuadratik kısmı, ÜST
           ÜÇGENSEL bir matris M_k ile temsil edilir; bu matriste
           YALNIZCA aynı katmana ait Oil x Oil çift indisleri (hem i
           hem j o katmanın Oil kümesinde) SIFIR bırakılır, geri kalan
           tüm (Vinegar x Vinegar ve Vinegar x Oil) terimler serbestçe
           rastgele seçilir. Bu kısıtlama, tek katmanlı Oil-Vinegar
           şemasındaki "Oil x Oil yasak" kuralının doğrudan katmanlı
           genellemesidir ve bu blok yapısı, imzalama sırasında
           Vinegar değişkenleri sabitlendiğinde Oil değişkenlerine
           göre DOĞRUSAL bir sistem elde edilmesini garanti eder
           (dolayısıyla katman katman çözülebilirliği sağlar). Ayrıca
           her polinoma rastgele bir doğrusal kısım (yalnızca o
           katmana kadar olan değişkenler üzerinden) ve rastgele bir
           sabit terim eklenir.

        3. Açık Anahtar (P):
           Girdi değişkenlerine T afin dönüşümü (y = T_mat * x + T_vec)
           sembolik olarak uygulanarak F_composed = F(T(x)) elde
           edilir; ardından çıktı tarafında S afin dönüşümü
           (P_vec = S_mat * F_composed + S_vec) uygulanarak nihai açık
           anahtar polinomları (Public_Key) elde edilir. Bu bileşim,
           merkezi dönüşümün katman katman kısıtlı blok yapısını T ve
           S'nin doğrusal karıştırma etkisiyle görünürde tamamen
           gizler -- Rainbow'un trapdoor mekanizmasının özü budur.

        Yan Etkiler
        -----------
        self.T_mat, self.T_vec, self.S_mat, self.S_vec,
        self.Central_Polys ve self.Public_Key öznitelikleri bu metot
        tarafından (yeniden) doldurulur. Metot birden fazla kez
        çağrılırsa önceki anahtar bileşenleri (Central_Polys,
        Public_Key) temizlenip yeniden üretilir. Süreç boyunca
        ayrıntılı ara çıktılar (matrisler, polinomlar, katman bazlı
        Oil x Oil blok kontrolü) ekrana yazdırılır.

        Dönüş Değeri
        ------------
        None
        """
        # generate_keys() birden fazla kez çağrılırsa eski anahtar bileşenleri temizlenir.
        self.Central_Polys = []
        self.Public_Key = []

        print("\n" + "*"*30 + " ADIM 1: ANAHTAR ÜRETİMİ " + "*"*30)

        # --- 1. AFİN DÖNÜŞÜMLER ---
        # T_mat/T_vec: girdi (değişken) tarafındaki gizli afin dönüşüm;
        # merkezi dönüşümün katman yapısını hangi değişken
        # kombinasyonlarının oluşturduğunu saldırgandan gizler.
        while True:
            self.T_mat = random_matrix(self.F, self.n, self.n)
            if self.T_mat.is_invertible(): break
        self.T_vec = random_vector(self.F, self.n)

        # S_mat/S_vec: çıktı (denklem) tarafındaki gizli afin dönüşüm;
        # merkezi denklemlerin hangi doğrusal kombinasyonlarının açık
        # anahtar denklemlerini oluşturduğunu gizler.
        while True:
            self.S_mat = random_matrix(self.F, self.m, self.m)
            if self.S_mat.is_invertible(): break
        self.S_vec = random_vector(self.F, self.m)

        print(f"\n[1.1] Gizli Afin Dönüşümler:")
        print(f"   T Matrisi ({self.n}x{self.n}):\n{self.T_mat}")
        print(f"   T Vektörü: {self.T_vec}")
        print(f"   S Matrisi ({self.m}x{self.m}):\n{self.S_mat}")
        print(f"   S Vektörü: {self.S_vec}")

        # --- 2. MERKEZ Dönüşüm (F) ---
        print(f"\n[1.2] Gizli Merkezi Dönüşüm F(x) (Katman Katman)")

        global_idx = 0

        for layer in self.layers:
            lid = layer['id']
            v_idx = list(layer['v_indices'])
            o_idx = list(layer['o_indices'])

            print(f"\n   >>> KATMAN {lid}:")
            print(f"       Vinegar Değişkenleri: x{v_idx[0]}..x{v_idx[-1]} (Toplam {len(v_idx)})")
            print(f"       Oil Değişkenleri:     x{o_idx[0]}..x{o_idx[-1]} (Toplam {len(o_idx)})")
            print(f"       KURAL: Oil x Oil ({len(o_idx)}x{len(o_idx)}) bloğu SIFIR olmalı!")

            for _ in range(layer['num_eqs']):
                # A) Matris Oluştur (Üst Üçgen)
                # M_k, bu katmanın merkezi polinomunun kuadratik kısmını
                # temsil eden üst üçgensel matristir. Yalnızca bu katmanın
                # Oil x Oil çift indisleri (hem i hem j o katmanın Oil
                # kümesinde) SIFIR bırakılır; bu, o katmanın Oil
                # değişkenlerinin BİRBİRİYLE çarpılmadığı, dolayısıyla
                # Vinegar değişkenleri sabitlendiğinde Oil'e göre
                # DOĞRUSAL kalınacağı anlamına gelir.
                M_k = matrix(self.F, self.n, self.n)

                # İzin verilen terimler:
                # - Vinegar x vinegar
                # - Vinegar x oil
                # Yasaklanan terimler:
                # - Aynı katmana ait oil x oil terimleri
                for i in range(o_idx[-1] + 1):
                    for j in range(i, o_idx[-1] + 1):
                        if not (i in o_idx and j in o_idx):
                            M_k[i, j] = self.F.random_element()

                # O x O bloğu (i in o_indices, j in o_indices) SIFIR KALIR.

                # B) Polinomu Türet
                quad_part = self.vars * M_k * self.vars

                # Lineer ve Sabit
                # Doğrusal kısım yalnızca bu katmana kadar olan
                # değişkenler (Vinegar + bu katmanın Oil'i) üzerinden
                # tanımlanır; daha sonraki katmanların değişkenleri bu
                # polinomda hiç görünmez (katmanlı yapının bir gereği).
                lin_part = self.R(0)
                for i in range(o_idx[-1] + 1): # Sadece bu katmana kadar olan değişkenler
                    lin_part += self.F.random_element() * self.vars[i]
                const_part = self.F.random_element()

                poly = quad_part + lin_part + const_part
                self.Central_Polys.append(poly)

                print(f"\n      Denklem f_{global_idx}:")
                print(f"      Polinom: {poly}")
                print(f"      Matris M_{global_idx} (Sağ Alt Blok Sıfır mı?):")
                print(M_k)

                # Kontrol
                # Bu denetim, M_k inşasının doğruluğunu gözle teyit
                # etmek amacıyla eklenmiştir: Oil x Oil alt bloğunun
                # (submatrix) gerçekten sıfır matris olduğu doğrulanır.
                oil_block = M_k.submatrix(o_idx[0], o_idx[0], len(o_idx), len(o_idx))
                if oil_block.is_zero():
                    print("      [ONAY] Oil x Oil bloğu temiz.")
                else:
                    print("      [HATA] Oil bloğunda eleman var!")

                global_idx += 1

        # --- 3. AÇIK ANAHTAR (P) ---
        print(f"\n[1.3] Açık Anahtar P(x) = S( F( T(x) ) )")

        # A) T(x)
        # Girdi tarafındaki afin dönüşüm sembolik olarak uygulanır:
        # y = T_mat * x + T_vec. Bu, merkezi polinomların hangi
        # DEĞİŞKEN KOMBİNASYONLARI üzerinden değerlendirileceğini
        # saldırgandan gizleyen ilk maskeleme katmanıdır.
        y_vec = self.T_mat * self.vars + self.T_vec
        sub_dict = {self.vars[i]: y_vec[i] for i in range(self.n)}

        # B) F(T(x))
        F_composed = [f.subs(sub_dict) for f in self.Central_Polys]

        # C) S( F_composed )
        # Çıktı tarafındaki afin dönüşüm uygulanır: P = S_mat * F(T(x)) + S_vec.
        # Bu, merkezi denklemlerin hangi DOĞRUSAL KOMBİNASYONLARININ açık
        # anahtar denklemlerini oluşturduğunu gizleyen ikinci maskeleme
        # katmanıdır; sonuçta F'nin katman katman kısıtlı blok yapısı
        # açık anahtar polinomlarında görünürde tamamen kaybolur.
        F_vec = vector(self.R, F_composed)
        P_vec = self.S_mat * F_vec + self.S_vec
        self.Public_Key = list(P_vec)

        print(f"   -> Açık Anahtar Polinomları ve Matrisleri:")

        for idx, p_poly in enumerate(self.Public_Key):
            print(f"\n      P_{idx} (Public):")
            print(f"      Polinom: {p_poly}")

            M_pub = self.get_poly_matrix(p_poly)
            print(f"      Matris M_pub_{idx} (Tamamen Karışık):")
            print(M_pub)

    def sign(self, message_vec):
        """
        Verilen bir mesaj vektörü için, gizli anahtar bilgisini
        (T, S afin dönüşümleri ve katman katman kısıtlanmış merkezi
        dönüşüm F) kullanarak Rainbow imzasını üretir.

        Algoritmanın Adımları
        -----------------------
        1. Hedefin Geri Çekilmesi (S^{-1}):
           y_target = S_mat^{-1} * (message_vec - S_vec) hesaplanarak,
           açık anahtar denklemlerinin çıktı tarafındaki afin
           maskelemesi kaldırılır; artık y_target, merkezi dönüşüm
           F'nin DOĞRUDAN ulaşması gereken hedef değerlerdir.

        2. Rastgele Başlangıç Vinegar Seçimi:
           İlk v1 değişken (hiçbir katmanda Oil olarak çözülmeyen
           başlangıç Vinegar kümesi) rastgele seçilir.

        3. Katman Katman Doğrusal Çözüm:
           Her katman için, önceki tüm değişkenler (kendi katmanının
           Vinegar'ı) BİLİNEN kabul edilip merkezi polinomlarda yerine
           konur; bu, Oil x Oil teriminin sıfır olması sayesinde geri
           kalan Oil değişkenlerine göre DOĞRUSAL bir sistem (A * oil =
           b) doğurur. Bu sistem Sage'in solve_right metoduyla
           çözülür; çözüm bulunursa değerler vals dizisine yazılıp bir
           sonraki katmana geçilir. Sistem çözülemezse (tekil/tutarsız
           matris), TÜM işlem terk edilip yeni bir rastgele başlangıç
           Vinegar kümesiyle yeniden denenir (RETRY mekanizması; bu,
           Oil-Vinegar ailesi şemalarında beklenen, güvenliği
           etkilemeyen bir durumdur).

        4. Girdi Maskelemesinin Kaldırılması (T^{-1}):
           Tüm katmanlar başarıyla çözüldüğünde, elde edilen merkezi
           çözüm vektörü z'ye T_mat^{-1} uygulanarak
           signature = T_mat^{-1} * (z - T_vec) ile gerçek imza elde
           edilir.

        Ön Koşullar
        ------------
        generate_keys() metodunun bu metottan önce çağrılmış olması
        gerekir (_check_keys_generated aracılığıyla denetlenir).
        message_vec'in uzunluğu self.m'ye eşit olmalıdır.

        Parametreler
        ------------
        message_vec : vector
            Uzunluğu self.m olan, F_q üzerinde tanımlı, imzalanacak
            (tipik olarak bir hash fonksiyonundan gelen) mesaj vektörü.

        Dönüş Değeri
        ------------
        vector veya None
            Başarılı olursa, uzunluğu self.n olan geçerli bir Rainbow
            imzası; en fazla 100 denemede hiçbir başlangıç Vinegar
            kümesi tüm katmanlarda çözülebilir bir sistem üretmezse
            None.

        Fırlatılan İstisnalar
        -----------------------
        RuntimeError
            Anahtarlar henüz üretilmemişse (_check_keys_generated
            aracılığıyla).
        ValueError
            message_vec'in uzunluğu self.m'ye eşit değilse.
        """
        self._check_keys_generated()

        if len(message_vec) != self.m:
            raise ValueError(f"Mesaj vektörünün uzunluğu m={self.m} olmalıdır.")

        print("\n" + "*"*30 + " ADIM 2: İMZALAMA SÜRECİ " + "*"*30)
        print(f"   İmzalanacak Mesaj (h): {message_vec}")

        # 1. S^-1
        # Açık anahtarın çıktı tarafındaki afin maskelemesi kaldırılır;
        # y_target artık merkezi dönüşüm F'nin doğrudan ulaşması gereken
        # hedef değerleri temsil eder.
        y_target = self.S_mat.inverse() * (message_vec - self.S_vec)
        print(f"\n[2.1] Hedef Değerler y = S^-1(h): {y_target}")

        attempts = 0
        while attempts < 100:
            attempts += 1
            print(f"\n   --- Deneme {attempts} ---")

            # A) Başlangıç Vinegar Seçimi
            # x0 .. xv1-1 arası değişkenler
            # Bu değişkenler hiçbir katmanda Oil olarak çözülmez;
            # yalnızca rastgele seçilip sabit tutulur ve her katmanın
            # doğrusal sistemine "bilinen" olarak girerler.
            vals = [self.F(0)] * self.n
            initial_vinegar = random_vector(self.F, self.v1)

            for i in range(self.v1):
                vals[i] = initial_vinegar[i]

            print(f"      1. Rastgele Başlangıç Vinegar (Layer 0): {initial_vinegar}")

            # B) Katman Katman Çözüm
            all_solved = True
            eq_idx = 0

            for layer in self.layers:
                lid = layer['id']
                v_idx = layer['v_indices']
                o_idx = layer['o_indices'] # Bilinmeyenler (Oil)
                num = layer['num_eqs']

                print(f"      >>> Katman {lid} Çözülüyor (Bilinmeyenler: x{o_idx[0]}..x{o_idx[-1]})...")

                # Lineer Sistem: A * oil = b
                coeffs = []
                consts = []

                # Bilinenleri (önceki katmanlar + initial vinegar) yerine koy
                # Bu katmanın Vinegar kümesi, indeksi o_idx[0]'dan küçük
                # tüm değişkenlerdir; bunlar önceki adımlarda (ya
                # başlangıç Vinegar olarak ya da önceki katmanların Oil
                # çözümü olarak) zaten belirlenmiştir.
                known_dict = {self.vars[k]: vals[k] for k in range(o_idx[0])}

                for k in range(num):
                    poly = self.Central_Polys[eq_idx + k]
                    target = y_target[eq_idx + k]

                    # Vinegar'ları göm -> Lineer denklem kalır
                    # Bilinen (Vinegar) değişkenler polinomda yerine
                    # konduğunda, Oil x Oil teriminin merkezi dönüşümde
                    # zaten sıfır olması sayesinde geriye yalnızca Oil
                    # değişkenlerine göre DOĞRUSAL bir ifade kalır.
                    f_linear = poly.subs(known_dict)

                    # a1*o1 + ... + C = target => a1*o1 = target - C
                    row = []
                    for oil_var_idx in o_idx:
                        row.append(f_linear.monomial_coefficient(self.vars[oil_var_idx]))

                    rhs = target - f_linear.constant_coefficient()

                    coeffs.append(row)
                    consts.append(rhs)

                # Matrisi Çöz
                A = matrix(self.F, coeffs)
                b = vector(self.F, consts)
                # ------------------------------------------------------------
                # LINEER SİSTEMİ YAZDIRMA
                # ------------------------------------------------------------
                print(f"\n      [Katman {lid}] Oil Değişkenleri İçin Lineer Sistem: A * oil = b")
                print(f"      Bilinmeyenler: {[self.vars[i] for i in o_idx]}")
                print(f"      A Matrisi:\n{A}")
                print(f"      b Vektörü: {b}")

                print("      Açık Denklem Sistemi:")

                for row_idx in range(A.nrows()):
                    equation_terms = []

                    for col_idx in range(A.ncols()):
                        coeff = A[row_idx, col_idx]

                        if coeff != 0:
                            oil_var_name = self.vars[o_idx[col_idx]]
                            equation_terms.append(f"({coeff})*{oil_var_name}")

                    lhs = " + ".join(equation_terms) if equation_terms else "0"
                    rhs = b[row_idx]

                    print(f"         Denklem {row_idx+1}: {lhs} = {rhs}")

                print("      ------------------------------------------------------------")
                try:
                    sol = A.solve_right(b)
                    print(f"          [BAŞARILI] Çözüm: {sol}")
                    # Değerleri kaydet
                    for k, val in enumerate(sol):
                        vals[o_idx[0] + k] = val

                    eq_idx += num

                except ValueError:
                    # A matrisi tekil (tersinir olmayan doğrusal sistem)
                    # ise bu katman çözülemez; bu durumda başlangıç
                    # Vinegar değerleri kötü bir seçim olmuş demektir ve
                    # tüm süreç yeni bir rastgele seçimle baştan
                    # denenmelidir.
                    print("          [HATA] Lineer sistem çözülemedi. Yeni vinegar seçiliyor...")
                    all_solved = False
                    break

            if all_solved:
                # C) T^-1
                # Tüm katmanlar başarıyla çözüldüğünde, merkezi
                # koordinat sistemindeki çözüm vektörü z, girdi
                # tarafındaki afin maskelemenin tersi (T^{-1})
                # uygulanarak gerçek (açık koordinat sistemindeki)
                # imzaya dönüştürülür.
                z_vec = vector(self.F, vals)
                print(f"\n      [TAMAMLANDI] Merkezi Çözüm z: {z_vec}")

                signature = self.T_mat.inverse() * (z_vec - self.T_vec)
                print(f"      [SONUÇ] T^-1 Uygulandı. İMZA (x): {signature}")
                return signature

        return None

    def verify(self, message, signature):
        """
        Verilen bir imzanın, açık anahtar polinomları aracılığıyla
        verilen mesaja karşılık gelip gelmediğini doğrular.

        Teorik Gerekçe
        ---------------
        Rainbow (ve genel olarak çok değişkenli kuadratik imza
        şemaları) tek yönlü bir kapı fonksiyonuna dayanır: P(x)'i
        (gizli anahtar bilgisi olmadan) HERHANGİ bir y hedefi için
        tersine çevirmek hesaplama açısından zordur, ancak P(x)'i
        belirli bir x noktasında DEĞERLENDİRMEK (ileri yön) kolaydır.
        Doğrulama, tam olarak bu kolay yönü kullanır: imza doğrudan
        açık anahtar polinomlarında yerine konur ve sonucun mesajla
        eşleşip eşleşmediği kontrol edilir; bu adım hiçbir gizli
        anahtar bilgisi gerektirmez.

        Ön Koşullar
        ------------
        generate_keys() metodunun bu metottan önce çağrılmış olması
        gerekir (_check_keys_generated aracılığıyla denetlenir).
        message'ın uzunluğu self.m'ye, signature'ın uzunluğu self.n'ye
        eşit olmalıdır.

        Parametreler
        ------------
        message : vector
            Uzunluğu self.m olan, F_q üzerinde tanımlı orijinal mesaj
            vektörü.
        signature : vector
            Uzunluğu self.n olan, F_q üzerinde tanımlı, doğrulanacak
            imza vektörü (tipik olarak sign() metodunun çıktısı).

        Dönüş Değeri
        ------------
        bool
            İmza, açık anahtar polinomlarında değerlendirildiğinde
            mesajla tam olarak eşleşiyorsa True; aksi halde False.

        Fırlatılan İstisnalar
        -----------------------
        RuntimeError
            Anahtarlar henüz üretilmemişse (_check_keys_generated
            aracılığıyla).
        ValueError
            message'ın uzunluğu self.m'ye veya signature'ın uzunluğu
            self.n'ye eşit değilse.
        """
        self._check_keys_generated()

        if len(message) != self.m:
            raise ValueError(f"Mesaj vektörünün uzunluğu m={self.m} olmalıdır.")

        if len(signature) != self.n:
            raise ValueError(f"İmza vektörünün uzunluğu n={self.n} olmalıdır.")

        print("\n" + "*"*30 + " ADIM 3: DOĞRULAMA " + "*"*30)
        val_dict = {self.vars[i]: signature[i] for i in range(self.n)}
        check = [p.subs(val_dict) for p in self.Public_Key]
        check_vec = vector(self.F, check)

        print(f"   P(İmza) = {check_vec}")
        print(f"   Mesaj   = {message}")
        return check_vec == message



In [3]:
# ============================================================================
# SENARYO: RAINBOW İMZA ŞEMASI İÇİN ÖĞRETİCİ SAGEMATH SİMÜLASYONU
# ============================================================================
# Bu örnekte GF(7) üzerinde iki katmanlı küçük bir Rainbow sistemi kurulmaktadır.
#
# Katman yapısı:
#   v_layers = [2, 4, 6]
#
# Bu şu anlama gelir:
#   Başlangıç vinegar değişkenleri: x0, x1
#   Katman 1 oil değişkenleri:      x2, x3
#   Katman 2 oil değişkenleri:      x4, x5
#
# Toplam değişken sayısı:
#   n = 6
#
# Toplam denklem sayısı:
#   m = n - v1 = 4
#
# Amaç:
#   1. Rainbow merkez dönüşümünü katmanlı şekilde üretmek.
#   2. Gizli afin dönüşümlerle açık anahtarı oluşturmak.
#   3. Katman katman çözüm yaparak imza üretmek.
#   4. Açık anahtar ile imzayı doğrulamak.
# ============================================================================

my_q = 7
my_layers = [2, 4, 6]

rainbow = RainbowSignatureScheme(q=my_q, v_layers=my_layers)
rainbow.generate_keys()

msg = random_vector(rainbow.F, rainbow.m)
sig = rainbow.sign(msg)



RAINBOW ŞEMASI: MATRİS VE POLİNOM ANALİZİ
Parametreler: GF(7)
Katman Yapısı: [2, 4, 6] (n=6, m=4)

****************************** ADIM 1: ANAHTAR ÜRETİMİ ******************************

[1.1] Gizli Afin Dönüşümler:
   T Matrisi (6x6):
[4 3 5 1 1 1]
[5 5 0 5 3 0]
[2 0 0 4 6 6]
[2 4 3 1 6 2]
[1 6 5 5 4 4]
[3 3 4 3 6 0]
   T Vektörü: (3, 1, 5, 5, 3, 4)
   S Matrisi (4x4):
[4 3 5 0]
[2 1 4 5]
[2 6 0 3]
[4 0 3 4]
   S Vektörü: (0, 5, 3, 5)

[1.2] Gizli Merkezi Dönüşüm F(x) (Katman Katman)

   >>> KATMAN 1:
       Vinegar Değişkenleri: x0..x1 (Toplam 2)
       Oil Değişkenleri:     x2..x3 (Toplam 2)
       KURAL: Oil x Oil (2x2) bloğu SIFIR olmalı!

      Denklem f_0:
      Polinom: x0^2 - 3*x0*x1 + 2*x1^2 + 3*x0*x2 + 2*x1*x2 - 2*x1*x3 - 3*x0 + x1 + 3*x2 + 2*x3 - 3
      Matris M_0 (Sağ Alt Blok Sıfır mı?):
[1 4 3 0 0 0]
[0 2 2 5 0 0]
[0 0 0 0 0 0]
[0 0 0 0 0 0]
[0 0 0 0 0 0]
[0 0 0 0 0 0]
      [ONAY] Oil x Oil bloğu temiz.

      Denklem f_1:
      Polinom: -2*x0^2 - 2*x0*x1 + 2*x0*x2 - 3